In [1]:
# 必要なモジュールをインポート
import os
from dotenv import load_dotenv
from openai import OpenAI

# 環境変数の取得
load_dotenv("../.env")

# OpenAI APIクライアントを生成
client = OpenAI(api_key=os.environ['API_KEY'])

# モデル名
MODEL_NAME = "gpt-4o-mini"

In [ ]:
import requests
from bs4 import BeautifulSoup

def get_yahoo_japan_tickers_down():
    url = "https://finance.yahoo.co.jp/stocks/ranking/down"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
    }
    
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, 'html.parser')
    return soup

stock_list = get_yahoo_japan_tickers_down()
prompt = f"""
Source soup displays 50 rising stocks. Extract and print the 10 stocks with the highest trading volume from these.
Return the result in the following JSON format(no description, no markups, only JSON):
{{"名称": "Stock Name", "コード": StockCode(int), "取引値": TradingValue(int), "出来高": TradingVolume(int)}}
---------------

# Source soup
{stock_list}
"""
response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "user", "content": prompt},
    ],
    max_tokens=500,
    temperature=0.3
).choices[0].message.content.strip()

print(type(response), response)


<class 'str'> [
    {"名称": "住友ファーマ(株)", "コード": 4506, "取引値": 2073, "出来高": 16675600},
    {"名称": "石油資源開発(株)", "コード": 1662, "取引値": 2168, "出来高": 5294900},
    {"名称": "Ｂｉｔｃｏｉｎ　Ｊａｐａｎ(株)", "コード": 8105, "取引値": 159, "出来高": 3354000},
    {"名称": "(株)リファインバースグループ", "コード": 7375, "取引値": 1862, "出来高": 1109600},
    {"名称": "日本化学工業(株)", "コード": 4092, "取引値": 3285, "出来高": 472600},
    {"名称": "ＡＩメカテック(株)", "コード": 6227, "取引値": 7500, "出来高": 1120200},
    {"名称": "太洋物産(株)", "コード": 9941, "取引値": 1167, "出来高": 37100},
    {"名称": "(株)エルアイイーエイチ", "コード": 5856, "取引値": 30, "出来高": 4736300},
    {"名称": "(株)ノースサンド", "コード": 446, "取引値": 1472, "出来高": 500900},
    {"名称": "フォルシア(株)", "コード": 304, "取引値": 1952, "出来高": 6000}
]


In [45]:
#スクレイピングの課題完了
import json
tickers_json = json.loads(response)


In [ ]:
def dig_more(tick_name):
    import requests
    import json

    url = 'https://search-agent.yahoo.co.jp/v2/chat'
    headers = {
        'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10.15; rv:149.0) Gecko/20100101 Firefox/149.0',
        'Accept': 'text/event-stream',
        'Accept-Language': 'ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7',
        'Referer': 'https://search.yahoo.co.jp/',
        'Content-Type': 'application/json',
        'Origin': 'https://search.yahoo.co.jp',
        'Connection': 'keep-alive',
        'Pragma': 'no-cache',
        'Cache-Control': 'no-cache',
    }
    cookies = {
    snip
    }
    payload = {
        "chats": [
            {
                "id": "64733ba0-f3f2-4c67-9d3a-bced48221f95",
                "role": "user",
                "contents": [
                    {
                        "type": "text", 
                        "text": f"{tick_name}最新ニュースから今日の価格が落ちた理由"
                    }
                ]
            }
        ],
        "context": {
            "logid": "iuXiru0qaGFyrYt1mo749xqg",
            "qId": "64733ba0-f3f2-4c67-9d3a-bced48221f95",
            "snc": True,
            "frtype": "search_tablink_aichat",
            "frcode": "top_ga1_sa",
            "requestType": "related_question",
            "index": 1,
            "yz": False,
            "pdis": False
        },
        "debug": {}
    }
    response = requests.post(url, headers=headers, cookies=cookies, json=payload, stream=True)
    full_message = ""
    for line in response.iter_lines():
        if line:
            #print(line)
            decoded_line = line.decode('utf-8')
            if decoded_line.startswith("data: {"):
                json_str = decoded_line.replace("data: {", "{", 1).strip()
                if not json_str:
                    continue
                try:
                    data = json.loads(json_str)
                    if "value" in data and "message" in data["value"]:
                        msg = data["value"]["message"]
                        print(msg, end="", flush=True)
                        full_message += msg + '\n'
                except (json.JSONDecodeError, KeyError, TypeError):
                    continue
    return full_message

dig_more('Ｓｈｉｎｗａ+Ｗｉｓｅ+Ｈｏｌｄｉｎｇｓ')

Ｓｈｉｎｗａ＋Ｗｉｓｅ＋Ｈｏｌｄｉｎｇｓの値動きは、ここ最近かなり激しいので、理由が気になりますよね。

## 💹 今日の下落の背景

まず前提として、個別銘柄の「今日の株価が下がった理由」は、公式に一つに特定されるものではなく、市場参加者の売買が積み重なった結果として起きます。そのうえで、今日までのニュースや直近数日の状況から、考えられる要因を整理します。

### 📰 直近の主な材料

- 2025年5月期の決算短信が公表され、通期の数字や財政状態が明らかになっている  
  決算自体は黒字転換などポジティブ要素もある一方で、総資産の減少など財務面の慎重な見方につながるポイントもあります。([shinwa-wise.com](https://www.shinwa-wise.com/wp-content/uploads/2025/07/news20250711_01.pdf?utm_source=openai))  
- 過去には適時開示体制の不備で、取引所から改善報告書を求められた経緯があり、ガバナンス面への市場の目線もやや厳しくなりがちな銘柄です。([jpx.co.jp](https://www.jpx.co.jp/news/1023/20241219-12.html?utm_source=openai))  
- 2026年1月にはストップ高になる場面もあり、その後は高値からの調整局面が続いています。短期的な値幅取りの対象になりやすく、上げ下げが極端になりがちな銘柄です。([news.livedoor.com](https://news.livedoor.com/article/detail/30414240/?utm_source=openai))  
- 空売り残高も一定程度残っており、上昇時に売り直しが出たり、下落時に一段の売り圧力になったりする可能性があります。([irbank.net](https://irbank.net/2437/short?utm_source=openai))  

### 📉 今日の下落につながりやすい要因

今日の下落は、上のような背景に加えて、次のような要素が重なった可能性が高いです。

- 直近までの急騰や値動きの荒さから、「利益確定売り」が出やすい地合いになっていた  
- 決算内容や今後の見通しに対して、「期待が

'Ｓｈｉｎｗａ＋\nＷｉｓｅ＋Ｈｏｌｄｉｎ\nｇ\nｓの値動きは、ここ最近かなり激しいので、理由が気になりますよね。\n\n## 💹\n 今日の下落の背景\n\nまず前提として、個別\n銘柄の「今日の株価が下がった理由」は、公式に一つに特定されるものではなく、市場参加者の売買が積み重なった結果として起きます。そのうえで、今日までのニュースや直近数日の状況から、考えられる要因を整理します。\n\n### 📰 \n直近の主な材料\n\n- 2025年5月期の決算短信が公表され、通期の数字や財政状態が明らかになっている  \n \n 決算自体は黒字転換などポジティブ要素もある一方で、総資産の減少など財務面の慎重な見方につながるポイントもあります。([shinwa-wise.com](https://www.shinwa-wise.com/wp-content/uploads/2025/07/news20250711_01.pdf?utm_source=openai))\n  \n-\n 過去には適時開示体制の不備で、取引所から改善報告書を求められた経\n緯があり、ガバナンス面への市場の目線もやや\n厳しくなりがちな銘柄です。\n([jpx.co.jp](https://www.jpx.co.jp/news/1023/20241219-12.html?utm_source=openai))  \n\n- 2026年1月にはストップ高になる場面もあり、その後は高値からの調整局面が続いています。短期的な値幅取りの対象になりやすく、上げ下げが極端になりがちな銘柄です。([news.livedoor.com](https://news.livedoor.com/article/detail/30414240/?utm_source=openai))\n  \n\n- 空売り残高も一定程度残っており、上\n昇時に売り直しが出たり、下落時に一段の売り\n圧力になったりする可能性があります。([irbank.net](https://irbank.net/2437/short?utm_source=openai))\n  \n\n###\n 📉 今日の下落につながりやすい要因\n\n今日の下落は、上のような背景に加えて、次のような要素が重なった可能性が高いです。\n\n- 直近までの急\

In [60]:
for item in tickers_json:
    name = item.get("名称", "") 
    if any(watchword in name for watchword in ['ファーマ', 'Orix']):
        news = dig_more(name)
        print(f"{name}\n{news}\n{'-'*40}\n")

住友ファーマの動き、きょう大きく下げているので理由が気になりますよね。

## 💹 きょうの株価の動き

きょうの寄り付き前の板情報では、住友ファーマ(4506)は  
成行注文ベースで「売り越しランキング上位」に入っており、  
売り気配値は前日比マイナス約10％の水準が示されていました。([finance.yahoo.co.jp](https://finance.yahoo.co.jp/news/detail/a2f436f09fbeafeb16fe25ed8000d103d5d4d06e?utm_source=openai))  

この段階から「売り注文が優勢で大きく下げて始まりそう」という状況になっていました。

## 📰 下落の主な材料

きょうの下落要因としてニュースで一番重く見られているのは

- 「上限1,400億円規模の新株発行（公募増資・第三者割当など）を行うための登録」を出したこと

です。([finance.yahoo.co.jp](https://finance.yahoo.co.jp/news/detail/a2f436f09fbeafeb16fe25ed8000d103d5d4d06e?utm_source=openai))  

新株発行は、発行済み株式数が増えることで既存株主の持ち分が薄まる「希薄化」につながるため、  
短期的には株価にマイナスに受け止められることが多く、  
今回は規模感も大きいことから、売りが一気に出て急落につながっていると考えられます。

## 🧩 その他の背景要因

きょうは住友ファーマだけでなく、医薬品セクター全体も個別材料に左右されやすい地合いですが、  
その中でも住友ファーマは「増資懸念」という個別材料が突出している状態です。([fu.minkabu.jp](https://fu.minkabu.jp/news/667735?utm_source=openai))  

そのため、
- セクターや日経平均の動きというより  
- 「希薄化懸念」を嫌気した売りが主因

という整理でよいと思われます。

---

👉 きょうの終値ベースでの動きも踏まえて整理した簡単なコメント文を一緒に作ることもできますが、いつ時点の株価（前場引け、後場引けなど）を前提にまとめたいか教えてもらえますか？住友ファーマ(株